# I2C device initialization

In [ ]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc8583 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8583", revision="1p1", emulator="ch341", logging=False, is_gui=False)

# Preset

In [ ]:
def preset():
    ic.SS_TIMEOUT = 3
    ic.FREQ_SHIFT = 0
    ic.FSW_SET = 0
    ic.SS_TIMEOUT = 0
    ic.SET_IBAT_SNS_RES = 0
    ic.PMID_IN_RANGE_DIS = 0
    ic.VEXT_SHUTDOWN_SET = 0
    ic.STANDBY_MODE_SET = 1

    ic.VEXT_OVP = 14 # 25V
    ic.IIN_OCP = 0xC1

    ic.IIN_UCP_DIS = 1

    ic.VBAT_OVP = 0xFF
    ic.VIN_OVP = 0xC # 24.6V
    ic.VOUT_OVP = 3 # 5.3V

    ic.C1P2OUT_OVP = 7
    ic.C1P2OUT_UVP = 7

    ic.IIN_OCP_DG = 3
    ic.VBAT_OVP_DIS = 1
    ic.IIN_REG_DIS = 1
    ic.VBAT_REG_DIS = 1
    ic.IBAT_REG_DIS = 1
    ic.IBAT_OCP_DIS = 1
    ic.NTC_FLT_DIS = 1
    ic.TDIE_REG_DIS = 1

    ic.SET_IBAT_SNS_HS = 0
    ic.SET_IBAT_SNS_RES = 1

    ic.ADC_EN = 1

In [ ]:
def decrease_iout(target_vout, iout_max, mode_n):

    list_rev_iout = list(i for i in range(1, iout_max+1))
    list_rev_iout.reverse()

    meas_vin  = dm1.voltage_100
    offset = abs(meas_vin - target_vout*mode_n) / iout_max

    for set_iout in list_rev_iout:
        
        estimate_rev_vin = target_vout* mode_n + offset*set_iout
        ps.ch1.vset = estimate_rev_vin
        ld.iset = set_iout
        print(f"VIN={estimate_rev_vin:.03f}, ILOAD={set_iout:.01f}")
        delay(0.5)

# FSW_SET[3:0] sweep
# IOUT : up to 16A / 1A steps
# VOUT target : 4.4V
# IIN_OCP = 0x81 (4.80375A)
# IIN_REG_DIS = 1
# VBAT_REG_DIS = 1


# efficiency measurement

In [ ]:
mode_n = 4

preset()

if mode_n == 4:
    ic.MODE = 0 # 4TO1 FWD
elif mode_n ==3:
    ic.MODE = 1 # 3TO1 FWD
elif mode_n ==2:
    ic.MODE = 2 # 2TO1 FWD
elif mode_n ==1:
    ic.MODE = 3 # 1TO1 FWD
else:
    raise("error")

target_vout = 4.4
init_vout = 4.3
bs.vset = init_vout

vin_init = init_vout * mode_n + 0.2
ps.ch1.cfg_all = vin_init, 5.3

ic.enable_vin_charging

In [ ]:
ic.status

# disconnect from the battery emulator

In [ ]:
if mode_n == 4:
    iout_max = 12
elif mode_n == 3:
    iout_max = 11
elif mode_n == 2:
    iout_max = 10
elif mode_n == 1:
    iout_max = 5

list_iout = list(i for i in range(1, iout_max+1))
ld.iset = list_iout[0]
ld.enable

limit_vout = 4.5
limit_vin  = 23

step_coarse = 0.04
step_fine   = 0.002
rin_path = abs((vin_init - dm1.voltage_100) / ps.ch1.current)
print(f"RIN={rin_path:.05f}ohm")
print(f"MODE : {mode_n} to 1 (MODE={ic.MODE})")

In [ ]:
print(f"RIN={rin_path:.05f}ohm")
print(f"MODE : {mode_n} to 1 (MODE={ic.MODE})")

estimate_vin = target_vout*mode_n + rin_path*(list_iout[0]/mode_n)
ps.ch1.vset = estimate_vin
set_vin = estimate_vin

ic.write_byte(0x42, 0xc0) # disable CONV_OCP
print(f"0x42 = {ic.read_byte(0x42):#04x} (bit7=CONV_OCP)")
dm2.set_nplc = 1

log.output_set_filename(f"{mode_n}to1_efficiency_15uFx4_retest")

for freq in range(0, 16):
    ic.FSW_SET = freq

    log.output_csv(["FSW_SET", "Frequency (kHz)", "VIN (V)", "IIN (A)", "VOUT (V)", "IOUT (A)", "Efficiency (%)", "PLoss (W)", "VIN_ADC", "VOUT_ADC", "IIN_ADC", "IBAT_ADC", "VEXT_ADC", "TDIE_ADC"])
    print(f"\nFSW_SET={freq}")

    for set_iload in list_iout:

        ld.iset = set_iload
        dummuy = dm2.voltage_10
        delay(0.5)

        while True:

            sts_chg = ic.CP_SWITCHING_STAT
            if sts_chg == 0:
                print(f"CP_SWITCHING_STAT={sts_chg}, start fault handler")
                decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
                break
            
            temp_vout = dm2.voltage_10
            diff = abs(temp_vout - target_vout)

            if (temp_vout > (target_vout-0.001)) and (temp_vout < (target_vout+0.001)):
                # print(f"break {temp_vout:.03f}, {target_vout:.03f}, {diff:.03f}")
                break

            if temp_vout < target_vout:
                
                if diff > 0.015:
                    set_vin += step_coarse
                    # print(f"low, coars, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
                else:
                    set_vin += step_fine
                    # print(f"low, fine, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
            else:
                if diff > 0.015:
                    set_vin -= step_coarse
                    # print(f"high, coars, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
                else:
                    set_vin -= (step_fine+0.001)
                    # print(f"high, fine, {set_vin:.03f}, {temp_vout:.03f}, {diff:.03f}")
            
            ps.ch1.vset = set_vin
            # print(f"final {set_vin:.03f}")
        
        meas_vin  = dm1.voltage_100
        meas_vout = dm2.voltage_10
        meas_iin  = ps.ch1.current
        meas_iout = ld.current
        meas_freq = ds.get_meas1 / 1000

        adc_vin  = ic.VIN_ADC * ic.lsb_vin
        adc_vout = ic.VOUT_ADC * ic.lsb_vout
        adc_iin  = ic.IIN_ADC * ic.lsb_iin
        adc_iout = ic.IBAT_ADC * ic.lsb_ibat
        adc_vext = ic.VEXT_ADC * ic.lsb_vext
        adc_tdie = ic.TDIE_ADC * ic.lsb_tdie
        
        efficiency = (meas_vout*meas_iout)/(meas_vin*meas_iin)*100
        ploss = (meas_vin*meas_iin) - (meas_vout*meas_iout)

        log.output_csv([freq, meas_freq, meas_vin, meas_iin, meas_vout, meas_iout, efficiency, ploss, adc_vin, adc_vout, adc_iin, adc_iout, adc_vext, adc_tdie])
        print(f"FREQ={meas_freq:.01f}kHz, VIN={meas_vin:.03f}V, IIN={meas_iin:.03f}A, VOUT={meas_vout:.03f}V, IOUT={meas_iout:.03f}A, EFFICIENCY={efficiency:.03f}% (TDIE={adc_tdie:.01f}C)")

        sts_chg = ic.CP_SWITCHING_STAT
        if sts_chg == 0:
            print(f"CP_SWITCHING_STAT={sts_chg}, start fault handler")
            decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
            break

    decrease_iout(target_vout=target_vout, iout_max=set_iload, mode_n=mode_n)
    log.output_csv([])

    sts_chg = ic.CP_SWITCHING_STAT
    if sts_chg == 0:
        break

In [ ]:
ic.status

In [ ]:
ic.disable_charging
ld.disable

In [ ]:
ic.status_adc

In [ ]:
hex(ic.read_byte(0))

In [ ]:
ic.i2c_scan()

In [ ]:
ic.update_i2c_address(110)